In [15]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

device = torch.device('cuda')

# include parent dir in python path
# sys.path.append('/media/carsen/ssd1/approxineuro/notebooks')
sys.path.append('/media/carsen/ssd1/github/oneshot')

In [17]:
from utils import data
mouse_id = 12
depth_separable = True
pool = True
clamp = True
use_30k = False # use all data recorded (>30k) or only 30k, performance will decrease if use only 30k.
data_path = '../data'

# change path to your own dm11 path
img_root = '/home/carsen/dm11_pachitariu/data/STIM/'
# weight_path = '/home/carsen/dm11_cluster/fengtongd/Desktop/github/oneshot/weights'
weight_path = '../weights/'

In [18]:
# load images
img = data.load_images(img_root, file=os.path.join(img_root, data.img_file_name[mouse_id]), crop=True)
nimg, Ly, Lx = img.shape
print('img: ', img.shape, img.min(), img.max(), img.dtype)

raw image shape:  (68032, 66, 264)
cropped image shape:  (68032, 66, 130)
image mean:  126.71216
image std:  61.42324
img:  (68032, 66, 130) -2.062935 2.088588 float32


In [19]:
# load neurons
fname = '%s_nat30k_%s.npz'%(data.db[mouse_id]['mname'], data.db[mouse_id]['datexp'])
spks, istim_train, istim_test, xpos, ypos, spks_rep_all = data.load_neurons(file_path = os.path.join(data_path, fname), mouse_id = mouse_id, fixtrain=use_30k)
n_stim, n_neurons = spks.shape
print('spks: ', spks.shape, spks.min(), spks.max())
print('spks_rep_all: ', len(spks_rep_all), spks_rep_all[0].shape)
print('istim_train: ', istim_train.shape, istim_train.min(), istim_train.max())
print('istim_test: ', istim_test.shape, istim_test.min(), istim_test.max())

# selec the subset of training set for hyperparam search
n_train = 2000
val_idxes = np.random.choice(np.where(istim_train)[0], n_train, replace=False)
istim_train = istim_train[val_idxes]
spks = spks[val_idxes]


loading activities from ../data/FX43_nat30k_2025_05_19.npz
spks:  (29500, 4180) -1.110223e-16 5.6614475
spks_rep_all:  500 (10, 4180)
istim_train:  (29500,) 532 30031
istim_test:  (500,) 32 531


In [20]:
# split train and validation set
itrain, ival = data.split_train_val(istim_train, train_frac=0.9)
print('itrain: ', itrain.shape, itrain.min(), itrain.max())
print('ival: ', ival.shape, ival.min(), ival.max())


splitting training and validation set...
there is currently no randomness in this function now, please make sure the istim_train is in random order!
itrain:  (1800,)
ival:  (200,)
itrain:  (1800,) 1 1999
ival:  (200,) 0 1990


In [21]:
spks, spks_rep_all = data.normalize_spks(spks, spks_rep_all, itrain)


normalizing neural data...


In [22]:
from utils import metrics
test_fev = metrics.fev_nan(spks_rep_all)
print('FEV (test): ', np.mean(test_fev))

valid_idxes = np.where(test_fev > 0.15)[0]
print('valid_idxes: ', len(valid_idxes))

FEV (test):  0.14340354963175786
valid_idxes:  1681


In [23]:
# ineur = np.arange(0, n_neurons) #np.arange(0, n_neurons, 5)
ineur = np.where(test_fev > 0.1)[0] # use only neurons with FEV > 0.15
spks_train = torch.from_numpy(spks[itrain][:,ineur])
spks_val = torch.from_numpy(spks[ival][:,ineur]) 
spks_rep_all = [spks_rep_all[i][:,ineur] for i in range(len(spks_rep_all))]
print('spks_train: ', spks_train.shape, spks_train.min(), spks_train.max())
print('spks_val: ', spks_val.shape, spks_val.min(), spks_val.max())

img_train = torch.from_numpy(img[istim_train][itrain]).to(device).unsqueeze(1) # change :130 to 25:100 
img_val = torch.from_numpy(img[istim_train][ival]).to(device).unsqueeze(1)
img_test = torch.from_numpy(img[istim_test]).to(device).unsqueeze(1)

print('img_train: ', img_train.shape, img_train.min(), img_train.max())
print('img_val: ', img_val.shape, img_val.min(), img_val.max())
print('img_test: ', img_test.shape, img_test.min(), img_test.max())

input_Ly, input_Lx = img_train.shape[-2:]

spks_train:  torch.Size([1800, 2341]) tensor(0.) tensor(22.7563)
spks_val:  torch.Size([200, 2341]) tensor(0.) tensor(22.8930)
img_train:  torch.Size([1800, 1, 66, 130]) tensor(-2.0629, device='cuda:0') tensor(2.0886, device='cuda:0')
img_val:  torch.Size([200, 1, 66, 130]) tensor(-2.0629, device='cuda:0') tensor(2.0886, device='cuda:0')
img_test:  torch.Size([500, 1, 66, 130]) tensor(-2.0629, device='cuda:0') tensor(2.0886, device='cuda:0')


In [24]:
train_real_responses = torch.ones_like(spks_train)
val_real_responses = torch.ones_like(spks_val)
# set nans to zero
train_real_responses[torch.isnan(spks_train)] = 0
val_real_responses[torch.isnan(spks_val)] = 0
spks_train[torch.isnan(spks_train)] = 0
spks_val[torch.isnan(spks_val)] = 0

In [25]:
# build model
from utils import model_builder, model_trainer
nlayers = 2
nconv1 = 16
nconv2 = 64
# model, in_channels = model_builder.build_model(NN=len(ineur), n_layers=nlayers, n_conv=nconv1, n_conv_mid=nconv2, pool=pool, depth_separable=depth_separable, input_Ly=input_Ly, input_Lx=input_Lx)
# model_name = model_builder.create_model_name(data.mouse_names[mouse_id], data.exp_date[mouse_id], n_layers=nlayers, in_channels=in_channels, clamp=clamp, ineuron=len(ineur))
# # weight_path = './checkpoints/'
# model_path = os.path.join(weight_path, 'fullmodel', data.mouse_names[mouse_id], model_name)
# print('modelpath: ', model_path)
# model = model.to(device)

input shape of readout:  (64, 33, 65)
model name:  FX43_051925_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_2341.pt
modelpath:  ../weights/fullmodel/FX43/FX43_051925_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_2341.pt


# grid search

In [26]:
# import itertools
# import os
# import torch
# import numpy as np
# from utils import model_trainer
# from utils import model_builder
# # --- Define search space ---
# learning_rates = [1e-3, 3e-4]
# weight_decays_core = [0.01, 0.1]
# l2_readouts = [0.01, 0.1]

# # --- Store results ---
# results = []

# # --- Search all combinations ---
# for lr, wd_core, l2_readout in itertools.product(learning_rates, weight_decays_core, l2_readouts):
#     print(f"Training with lr={lr}, wd_core={wd_core}, l2_readout={l2_readout}")

#     # --- Build model fresh ---
#     model, in_channels = model_builder.build_model(NN=len(ineur), n_layers=nlayers, n_conv=nconv1, n_conv_mid=nconv2, pool=pool, depth_separable=depth_separable, input_Ly=input_Ly, input_Lx=input_Lx)
#     model_name = model_builder.create_model_name(data.mouse_names[mouse_id], data.exp_date[mouse_id], n_layers=nlayers, in_channels=in_channels, clamp=clamp, ineuron=len(ineur))
#     # weight_path = './checkpoints/'
#     model_path = os.path.join(weight_path, 'fullmodel', data.mouse_names[mouse_id], model_name)
#     print('modelpath: ', model_path)
#     model = model.to(device)

#     # --- Train model ---
#     best_state_dict = model_trainer.monkey_train(
#         model,
#         spks_train, train_real_responses,
#         spks_val, val_real_responses,
#         img_train, img_val,
#         hs_readout=0,  # or tune later
#         l2_readout=l2_readout,
#         weight_decay_core=wd_core,
#         device=device
#     )

#     # --- Evaluate on val set ---
#     model.load_state_dict(best_state_dict)
#     model.eval()
#     _, val_varexp = model_trainer.monkey_val_epoch(model, img_val, spks_val, val_real_responses, batch_size=100, device=device)

#     mean_varexp = val_varexp.mean().item()
#     results.append({
#         'lr': lr,
#         'wd_core': wd_core,
#         'l2_readout': l2_readout,
#         'val_varexp': mean_varexp
#     })

#     print(f"→ val_varexp: {mean_varexp:.4f}")

# # --- Find best config ---
# best = max(results, key=lambda x: x['val_varexp'])
# print("\nBest hyperparams:")
# print(best)


# Bayesian optimization

In [27]:
import optuna

def objective(trial):
    lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
    wd_core = trial.suggest_loguniform("weight_decay_core", 1e-3, 1.0)
    l2_readout = trial.suggest_loguniform("l2_readout", 1e-3, 1.0)

    model, in_channels = model_builder.build_model(NN=len(ineur), n_layers=nlayers, n_conv=nconv1, n_conv_mid=nconv2, pool=pool, depth_separable=depth_separable, input_Ly=input_Ly, input_Lx=input_Lx)
    model_name = model_builder.create_model_name(data.mouse_names[mouse_id], data.exp_date[mouse_id], n_layers=nlayers, in_channels=in_channels, clamp=clamp, ineuron=len(ineur))
    # weight_path = './checkpoints/'
    model_path = os.path.join(weight_path, 'fullmodel', data.mouse_names[mouse_id], model_name)
    print('modelpath: ', model_path)
    model = model.to(device)

    best_state = model_trainer.monkey_train(
        model,
        spks_train, train_real_responses,
        spks_val, val_real_responses,
        img_train, img_val,
        hs_readout=0,
        l2_readout=l2_readout,
        weight_decay_core=wd_core,
        device=device,
        learning_rate=lr
    )

    model.load_state_dict(best_state)
    model.eval()
    _, varexp = model_trainer.monkey_val_epoch(model, img_val, spks_val, val_real_responses, batch_size=100, device=device)
    return -varexp.mean().item()  # minimize negative variance explained

study = optuna.create_study()
study.optimize(objective, n_trials=20)

[I 2025-05-28 20:51:14,764] A new study created in memory with name: no-name-63f77136-49c8-4921-95f2-6627a42cbb07


input shape of readout:  (64, 33, 65)
model name:  FX43_051925_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_2341.pt
modelpath:  ../weights/fullmodel/FX43/FX43_051925_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_2341.pt
0.0013414296440147388


/tmp/ipykernel_2374546/1443594691.py:4: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
/tmp/ipykernel_2374546/1443594691.py:5: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  wd_core = trial.suggest_loguniform("weight_decay_core", 1e-3, 1.0)
/tmp/ipykernel_2374546/1443594691.py:6: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  l2_readout = trial.suggest_loguniform("l2_readout", 1e-3, 1.0)


epoch 0, train_loss = 0.0098, val_loss = 0.0093, varexp_val = -0.0685, time 0.56s
epoch 1, train_loss = 0.0089, val_loss = 0.0089, varexp_val = 0.0021, time 1.12s
epoch 2, train_loss = 0.0088, val_loss = 0.0088, varexp_val = 0.0064, time 1.68s
epoch 3, train_loss = 0.0087, val_loss = 0.0088, varexp_val = 0.0129, time 2.24s
epoch 4, train_loss = 0.0086, val_loss = 0.0088, varexp_val = 0.0192, time 2.82s
epoch 5, train_loss = 0.0086, val_loss = 0.0087, varexp_val = 0.0244, time 3.41s
epoch 6, train_loss = 0.0085, val_loss = 0.0086, varexp_val = 0.0327, time 3.98s
epoch 7, train_loss = 0.0085, val_loss = 0.0086, varexp_val = 0.0385, time 4.55s
epoch 8, train_loss = 0.0084, val_loss = 0.0086, varexp_val = 0.0398, time 5.15s
epoch 9, train_loss = 0.0084, val_loss = 0.0086, varexp_val = 0.0427, time 5.74s
epoch 10, train_loss = 0.0083, val_loss = 0.0085, varexp_val = 0.0484, time 6.33s
epoch 11, train_loss = 0.0083, val_loss = 0.0085, varexp_val = 0.0493, time 6.92s
epoch 12, train_loss = 0.

[I 2025-05-28 20:51:47,249] Trial 0 finished with value: -0.06502684205770493 and parameters: {'lr': 0.0013414296440147388, 'weight_decay_core': 0.0022697193906215554, 'l2_readout': 0.0031500852781426837}. Best is trial 0 with value: -0.06502684205770493.


...

epoch 9, train_loss = 0.0079, val_loss = 0.0084, varexp_val = 0.0668, time 12.36s
Early stopping at epoch 9 due to no improvement in validation varexp.
input shape of readout:  (64, 33, 65)
model name:  FX43_051925_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_2341.pt
modelpath:  ../weights/fullmodel/FX43/FX43_051925_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_2341.pt
0.004000633799582268
epoch 0, train_loss = 0.0094, val_loss = 0.0090, varexp_val = -0.0081, time 1.23s
epoch 1, train_loss = 0.0088, val_loss = 0.0089, varexp_val = 0.0069, time 2.49s
epoch 2, train_loss = 0.0087, val_loss = 0.0088, varexp_val = 0.0168, time 3.71s
epoch 3, train_loss = 0.0086, val_loss = 0.0087, varexp_val = 0.0265, time 4.97s
epoch 4, train_loss = 0.0085, val_loss = 0.0086, varexp_val = 0.0361, time 6.21s
epoch 5, train_loss = 0.0084, val_loss = 0.0086, varexp_val = 0.0408, time 7.45s
epoch 6, train_loss = 0.0084, val_loss = 0.0086, varexp_val = 0.0442, time 8.68s
epoch 7, train_loss = 0.00

[I 2025-05-28 21:13:24,613] Trial 18 finished with value: -0.06641078740358353 and parameters: {'lr': 0.004000633799582268, 'weight_decay_core': 0.030608687278773243, 'l2_readout': 0.0011263926973543949}. Best is trial 11 with value: -0.07454837113618851.


epoch 9, train_loss = 0.0080, val_loss = 0.0084, varexp_val = 0.0657, time 12.15s
Early stopping at epoch 9 due to no improvement in validation varexp.
input shape of readout:  (64, 33, 65)
model name:  FX43_051925_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_2341.pt
modelpath:  ../weights/fullmodel/FX43/FX43_051925_2layer_16_64_clamp_sensorium_depthsep_pool_nneurons_2341.pt
0.0007301710716215295
epoch 0, train_loss = 0.0099, val_loss = 0.0098, varexp_val = -0.1667, time 1.31s
epoch 1, train_loss = 0.0094, val_loss = 0.0090, varexp_val = -0.0102, time 2.55s
epoch 2, train_loss = 0.0088, val_loss = 0.0089, varexp_val = 0.0029, time 3.77s
epoch 3, train_loss = 0.0087, val_loss = 0.0088, varexp_val = 0.0055, time 5.03s
epoch 4, train_loss = 0.0087, val_loss = 0.0088, varexp_val = 0.0088, time 6.23s
epoch 5, train_loss = 0.0087, val_loss = 0.0088, varexp_val = 0.0133, time 7.45s
epoch 6, train_loss = 0.0086, val_loss = 0.0087, varexp_val = 0.0206, time 8.65s
epoch 7, train_loss = 0.

[I 2025-05-28 21:15:01,340] Trial 19 finished with value: -0.06936027109622955 and parameters: {'lr': 0.0007301710716215295, 'weight_decay_core': 0.0023852353252863385, 'l2_readout': 0.0034939767785758615}. Best is trial 11 with value: -0.07454837113618851.


epoch 4, train_loss = 0.0080, val_loss = 0.0084, varexp_val = 0.0691, time 5.94s
Early stopping at epoch 4 due to no improvement in validation varexp.


In [28]:
print("Best trial:")
best_trial = study.best_trial

print(f"  Value (objective): {-best_trial.value:.4f}")  # negate if you returned -val_varexp
print("  Params:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")


Best trial:
  Value (objective): 0.0745
  Params:
    lr: 0.002810981518314036
    weight_decay_core: 0.0011958116794013951
    l2_readout: 0.0010818284653872753


In [35]:
import joblib
joblib.dump(study, "optuna_study_2layers.pkl")


['optuna_study_2layers.pkl']

In [26]:
import joblib
study = joblib.load("optuna_study_4layers.pkl")

print("Best trial:")
best_trial = study.best_trial

print(f"  Value (objective): {-best_trial.value:.4f}")  # negate if you returned -val_varexp
print("  Params:")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

Best trial:
  Value (objective): 0.0755
  Params:
    lr: 0.0026146339759175433
    weight_decay_core: 0.06758502313016607
    l2_readout: 0.12013247147550409


In [ ]:
lr = [0.006, 0.003, 0.003, 0.003]
weight_decay_core = [0.1, 0.001, 0.003, 0.06]
l2_readout = [0.001, 0.001, 0.1, 0.1]

In [27]:
import optuna.visualization as vis

# Plot optimization history
vis.plot_optimization_history(study).show()

# Plot parameter importance (which hyperparams matter most)
vis.plot_param_importances(study).show()

# Parallel coordinate plot (hyperparams vs. objective)
vis.plot_parallel_coordinate(study).show()
